# Phase 5 confirmation — 5 folds, the promotion rule, and the checkpoints to ship

Fold-0 screening ranked candidates; **it never promotes one**. This is the run that does, against
the rule pre-registered before any of it: the winner must beat run 1 on the **full 5-fold pooled
OOF, full 12-label macro** — the competition metric, not the six-label screening subset — with the
paired bootstrap 95% CI excluding zero.

Three tiers come out of this run, and they answer different questions:

1. **Pooled OOF vs pseudo-labels** (4,349 studies) — the promotion decision. Paired against run 1's
   own OOF arrays, same studies, same targets.
2. **Gold transfer** (58 held-out studies, real rubric labels) — the only tier scored against
   labels read from *images* rather than text, and empirically the LB predictor: run 1's 0.7252
   preceded an LB of 0.763.
3. **Per-label** — because the phase's open question (does the coronal plane plus corrected
   laterality rescue MCL?) is invisible in any macro.

It also writes the five fold checkpoints the submission kernel loads.

In [ ]:
import glob, hashlib, os, shutil, sys, time

GIT_SHA = 'phase5-confirm'
FOLDS_PRIMARY_V2_SHA256 = 'cace8c45733ee7920a442fa4ae3f1db4a0d76401504e4729b49562d5eab52047'

# ---------------------------------------------------------------------------
# THE CONFIG UNDER TEST. Set by hand from screen B's summary rather than read
# from it: which row wins can be a judgement call (a larger macro bought with a
# much wider train-val gap is not automatically the better model), and this run
# costs hours -- it should not silently confirm whatever sorted first.
# ---------------------------------------------------------------------------
MAX_SERIES   = 2       # screen B: coronal series added
USE_LATERALITY = True  # screen B: corrected sides (98.5% coverage)
USE_AUGMENT  = True    # screen A: train 0.979 vs val 0.836 at 8 epochs
EPOCHS       = 8       # screen A winner; 12 was worse
N_SLICES, BATCH_SIZE, LR = 16, 8, 1e-4
CONFIG_HASH = (f'phase5_confirm_efficientnet_b0_s{MAX_SERIES}x{N_SLICES}_e{EPOCHS}'
               f'_onecycle_amp_lat{int(USE_LATERALITY)}_aug{int(USE_AUGMENT)}')

SRC = glob.glob('/kaggle/input/**/rsna-knee-src', recursive=True)[0]
COMP_DIR = glob.glob('/kaggle/input/**/rsna-knee-abnormality-detection', recursive=True)[0]
PREPPED_DIRS = sorted(glob.glob('/kaggle/input/**/prepped', recursive=True))
assert len(PREPPED_DIRS) == 4

uid_to_npz = {}
for d in PREPPED_DIRS:
    for p in sorted(glob.glob(os.path.join(d, '*.npz'))):
        uid_to_npz[os.path.splitext(os.path.basename(p))[0]] = p
NPZ_ROOT = '/kaggle/working/prepped_all'
os.makedirs(NPZ_ROOT, exist_ok=True)
for uid, p in uid_to_npz.items():
    dst = os.path.join(NPZ_ROOT, f'{uid}.npz')
    if not os.path.exists(dst):
        os.symlink(p, dst)

PKG = '/kaggle/working/knee'
os.makedirs(PKG, exist_ok=True)
for fname in os.listdir(SRC):
    if fname.endswith('.py'):
        shutil.copy(os.path.join(SRC, fname), os.path.join(PKG, fname))
sys.path.insert(0, '/kaggle/working')
print(f'{len(uid_to_npz)} artifacts; config {CONFIG_HASH}')

In [ ]:
import inspect
import torch

assert torch.cuda.is_available()
for i in range(torch.cuda.device_count()):
    major, minor = torch.cuda.get_device_capability(i)
    print(f'GPU {i}: {torch.cuda.get_device_name(i)}, sm_{major}{minor}')
    assert (major, minor) >= (7, 0), 'need T4, not P100'
device = 'cuda'

from knee.train import train_one_epoch as _t
from knee.dataset import PreppedStudyDataset as _d
assert {'scaler', 'scheduler'} <= set(inspect.signature(_t).parameters)
assert {'sides', 'augment'} <= set(inspect.signature(_d.__init__).parameters)
print('src carries every Phase 5 change this run depends on')

In [ ]:
import numpy as np
import pandas as pd

from knee.infer import LABEL_COLUMNS
from knee.train import (Timer, evaluate, load_gold_holdout, log_experiment,
                        make_folds, train_one_epoch, train_val_split)
from knee.dataset import PreppedStudyDataset
from knee.metrics import macro_auc, paired_macro_auc_delta, per_label_auc
from knee.model import KneeModel

train_df = pd.read_csv(f'{COMP_DIR}/train.csv')
all_uids = sorted(train_df['StudyInstanceUID'].astype(str))
assert len(all_uids) == 4407

pseudo = pd.read_csv(glob.glob('/kaggle/input/**/pseudo_labels_qwen3_4b.csv', recursive=True)[0])
labels_all = pseudo[['StudyInstanceUID'] + [f'score_{l}' for l in LABEL_COLUMNS]].copy()
labels_all.columns = ['StudyInstanceUID'] + LABEL_COLUMNS
_vals = labels_all[LABEL_COLUMNS].to_numpy(dtype=float)
assert np.isfinite(_vals).all() and (_vals >= 0).all() and (_vals <= 1).all()
labels_all['StudyInstanceUID'] = labels_all['StudyInstanceUID'].astype(str)
labels_eval = labels_all.copy()
labels_eval[LABEL_COLUMNS] = (_vals >= 0.5).astype(float)

folds = make_folds(all_uids, n_folds=5, seed=0)
assert hashlib.sha256('\n'.join(f'{u},{folds[u]}' for u in sorted(folds)).encode()
                      ).hexdigest() == FOLDS_PRIMARY_V2_SHA256

gold_df = train_df[train_df['ACL'].notna()].reset_index(drop=True)
assert len(gold_df) == 58
gold_df[['StudyInstanceUID']].to_csv('gold_tmp.csv', index=False)
holdout = load_gold_holdout('gold_tmp.csv')

SIDES = None
if USE_LATERALITY:
    lat = pd.read_csv(glob.glob('/kaggle/input/**/laterality_geometry_check.csv',
                                recursive=True)[0])
    def _final(r):
        if r['tag_side'] in ('L', 'R'):
            return r['tag_side']
        return r['geom_side'] if r['geom_side'] in ('L', 'R') else None
    lat['final'] = lat.apply(_final, axis=1)
    SIDES = {str(u): s for u, s in zip(lat['StudyInstanceUID'], lat['final'])
             if s in ('L', 'R')}
    assert len(SIDES) == 4341, len(SIDES)
    print(f'corrected sides: {len(SIDES)}/4407')

labeled_uids = [u for u in all_uids if u not in holdout]
assert len(labeled_uids) == 4349
row_of = {u: i for i, u in enumerate(labeled_uids)}
oof_true = np.full((len(labeled_uids), len(LABEL_COLUMNS)), np.nan)
oof_pred = np.full_like(oof_true, np.nan)

In [ ]:
def loader(uids, labels_df, shuffle=False, augment=False):
    ds = PreppedStudyDataset(uids, NPZ_ROOT, labels_df=labels_df, n_slices=N_SLICES,
                             max_series=MAX_SERIES, sides=SIDES, augment=augment)
    return torch.utils.data.DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle)

EXPERIMENTS_CSV = '/kaggle/working/experiments.csv'
_HEADER = ('date,git_sha,config_hash,hypothesis,fold_set,seed,acl_auc,mcl_auc,'
           'medial_meniscus_auc,lateral_meniscus_auc,medial_oa_auc,lateral_oa_auc,'
           'pf_oa_auc,effusion_auc,synovitis_auc,bakers_auc,contusion_auc,fracture_auc,'
           'macro_auc,paired_delta,train_minutes,inference_seconds,promoted')
with open(EXPERIMENTS_CSV, 'w') as f:
    f.write(_HEADER + '\n')

HYPOTHESIS = ('Phase 5 confirmation of the screened config on all 5 folds: '
              f'max_series={MAX_SERIES} laterality_v2={USE_LATERALITY} augment={USE_AUGMENT} '
              f'{EPOCHS}ep OneCycle AMP, vs run 1 (1 series, 1 epoch, flat LR, fp32)')

for fold in range(5):
    tr, va = train_val_split(folds, val_fold=fold, exclude_uids=holdout)
    assert not (set(tr) & set(va)) and not (set(tr) | set(va)) & holdout

    tl = loader(tr, labels_all, shuffle=True, augment=USE_AUGMENT)
    vl = loader(va, labels_eval)

    torch.manual_seed(fold)
    model = KneeModel(backbone_name='efficientnet_b0', num_labels=len(LABEL_COLUMNS),
                      pretrained=True).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    scaler = torch.amp.GradScaler('cuda', enabled=True)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, epochs=EPOCHS,
                                                steps_per_epoch=len(tl))
    timer = Timer()
    for ep in range(EPOCHS):
        with timer:
            loss = train_one_epoch(model, tl, opt, device=device, scaler=scaler, scheduler=sched)
        print(f'  fold {fold} epoch {ep+1}/{EPOCHS} loss {loss:.4f}')

    t0 = time.time()
    y_true, y_pred = evaluate(model, vl, device=device)
    infer_s = time.time() - t0
    for uid, yt, yp in zip(va, y_true, y_pred):
        oof_true[row_of[uid]] = yt
        oof_pred[row_of[uid]] = yp

    aucs = per_label_auc(y_true, y_pred)
    fold_macro = macro_auc(y_true, y_pred)
    print(f'fold {fold}: macro {fold_macro:.4f}  {timer.minutes:.1f} min')
    log_experiment(EXPERIMENTS_CSV, git_sha=GIT_SHA, config_hash=CONFIG_HASH,
                   hypothesis=HYPOTHESIS, fold_set=f'primary_v2_fold{fold}', seed=fold,
                   per_label_auc={l: float(a) for l, a in zip(LABEL_COLUMNS, aucs)},
                   macro_auc=float(fold_macro), paired_delta=0.0,
                   train_minutes=timer.minutes, inference_seconds=infer_s, promoted=False)

    # pretrained=False on reload later; save weights only
    torch.save(model.state_dict(), f'/kaggle/working/knee_phase5_fold{fold}.pt')
    del model, opt, scaler, sched
    torch.cuda.empty_cache()

assert not np.isnan(oof_pred).any(), 'some study never landed in a validation fold'
np.save('/kaggle/working/oof_pred.npy', oof_pred)
np.save('/kaggle/working/oof_true.npy', oof_true)
np.save('/kaggle/working/oof_uids.npy', np.array(labeled_uids, dtype=object))

## Tier 1 — the promotion decision

Full 12-label macro over the pooled OOF, paired against run 1 on the same studies. This is the
competition's own metric, not the screening subset.

In [ ]:
oof_dir = os.path.dirname(glob.glob('/kaggle/input/**/knee-phase4-train/oof_pred.npy',
                                    recursive=True)[0])
base_pred = np.load(f'{oof_dir}/oof_pred.npy')
base_uids = np.load(f'{oof_dir}/oof_uids.npy', allow_pickle=True).astype(str)
assert list(base_uids) == list(labeled_uids), 'run 1 OOF rows do not line up with this run'

run1_macro = macro_auc(oof_true, base_pred)
new_macro = macro_auc(oof_true, oof_pred)
delta, lo, hi = paired_macro_auc_delta(oof_true, base_pred, oof_pred)
print(f'run 1 pooled OOF macro : {run1_macro:.4f}')
print(f'this run pooled OOF    : {new_macro:.4f}')
print(f'paired delta           : {delta:+.4f}  95% CI [{lo:+.4f}, {hi:+.4f}]')
PROMOTED = lo > 0
print('\nPROMOTION RULE:', 'PASS -- CI excludes zero' if PROMOTED else 'FAIL -- CI includes zero')

per = pd.DataFrame({
    'run1': per_label_auc(oof_true, base_pred),
    'phase5': per_label_auc(oof_true, oof_pred),
}, index=LABEL_COLUMNS)
per['delta'] = per['phase5'] - per['run1']
print('\n', per.round(4).to_string())

## Tier 2 — gold transfer, the LB predictor

The 58 studies with real rubric labels, held out of training on every fold. Run 1 scored 0.7252
here and then 0.763 on the leaderboard, so this is the number that forecasts the submission. It is
n=58 and cannot arbitrate between close configurations — it is reported, never used to select.

In [ ]:
gold_labels_df = gold_df[['StudyInstanceUID'] + LABEL_COLUMNS].copy()
gold_uids = gold_labels_df['StudyInstanceUID'].astype(str).tolist()
gold_loader = loader(gold_uids, gold_labels_df)

fold_preds = []
for fold in range(5):
    model = KneeModel(backbone_name='efficientnet_b0', num_labels=len(LABEL_COLUMNS),
                      pretrained=False).to(device)
    model.load_state_dict(torch.load(f'/kaggle/working/knee_phase5_fold{fold}.pt'))
    gt, gp = evaluate(model, gold_loader, device=device)
    fold_preds.append(gp)
    print(f'  fold {fold} gold macro {macro_auc(gt, gp):.4f}')
    del model
    torch.cuda.empty_cache()

gold_ensemble = np.mean(fold_preds, axis=0)
gold_macro = macro_auc(gt, gold_ensemble)
print(f'\nGOLD TRANSFER (5-model mean): {gold_macro:.4f}   [run 1: 0.7252 -> LB 0.763]')
np.save('/kaggle/working/gold_pred_ensemble.npy', gold_ensemble)

gold_per = pd.Series(per_label_auc(gt, gold_ensemble), index=LABEL_COLUMNS)
run1_gold = pd.Series({'ACL': 0.8051, 'MCL': 0.5034, 'Medial Meniscus': 0.6298,
                       'Lateral Meniscus': 0.7081, 'Medial OA': 0.7922, 'Lateral OA': 0.6132,
                       'PF OA': 0.6821, 'Effusion': 0.9391, 'Synovitis': 0.7802,
                       "Baker's": 0.7355, 'Contusion': 0.8084, 'Fracture': 0.7056})
cmp = pd.DataFrame({'run1_gold': run1_gold, 'phase5_gold': gold_per})
cmp['delta'] = cmp['phase5_gold'] - cmp['run1_gold']
print('\n', cmp.round(4).to_string())
print('\nMCL was 0.5034 (random) in run 1 -- the coronal+laterality hypothesis lives or dies here.')

In [ ]:
summary = pd.read_csv(EXPERIMENTS_CSV)
summary.loc[summary.index[-5:], 'paired_delta'] = delta
summary.loc[summary.index[-5:], 'promoted'] = PROMOTED
summary.to_csv(EXPERIMENTS_CSV, index=False)
print(f'pooled OOF {new_macro:.4f} (run 1 {run1_macro:.4f}, delta {delta:+.4f})')
print(f'gold transfer {gold_macro:.4f} (run 1 0.7252)')
print('promoted:', PROMOTED)